## About this notebook:

This notebook fixes one round's alignment by hand, for when 02's automatic alignment got it wrong. In napari you click the same cells in the anchor round and in the round to fix, and the round's transform is replaced with the one your clicks give.

Input:
- the well's outlines from 01 and transforms from 02
- `well` and `round_name`: the well and the round to fix, the round written as in `info_exp.csv` (e.g. `"R1"`). The anchor round can't be fixed: every other round is aligned onto it.

Output:
- `output_df/tmat_<well>.npy` with that round's transform replaced. The other rounds keep 02's transforms.

The last cells (optional) open napari again to check the saved alignment of every round in the well.

## Choose the experiment

Set `CONFIG` to the experiment's `experiment.json`. All folders and the well list come from it; create or edit it with `setup_experiment.ipynb`.

In [ ]:
from fouri import load_config

# This experiment's experiment.json (create or edit it with setup_experiment.ipynb)
CONFIG = r'Z:\PATH\TO\EXPERIMENT\experiment.json'

exp = load_config(CONFIG)

well = exp.well_list[0]  # change to the well to correct, e.g. 'D03'
round_name = "R1"  # the round to correct, as written in info_exp.csv

## Prepare for processing

In [ ]:
import napari
import numpy as np

from fouri.align import alignment_preview, manual_alignment_images, save_manual_transform, start_hand_alignment

In [ ]:
round_index, anchor_labels, round_labels = manual_alignment_images(exp, well, round_name)

## Line up the round by hand in napari

Napari opens with the two rounds on top of each other. Clicks alternate between them, and the text at the top of the viewer says which round to click next.

* Magenta = anchor round
* Green = round to fix
* White = cells that line up

1. Click a cell in the magenta round, then the same cell in the green round. Scroll to zoom in; hold Space and drag to move around.
2. Repeat for at least 3 cells spread across the image. From the second pair on, the green round moves onto the magenta one, and the top of the viewer shows how far apart the pairs still are.
3. When the cells show white across the whole image, run the next cell.

Clicked the wrong cell? Run the cell below again to start over.

In [ ]:
viewer = napari.Viewer()
anchor_layer = viewer.add_image(anchor_labels, blending="additive", colormap="magenta", name="Anchor Round")
round_layer = viewer.add_image(round_labels, blending="additive", colormap="green", name=f"{round_name} (to fix)")
start_hand_alignment(viewer, anchor_layer, round_layer)

In [ ]:
napari_affine = np.asarray(round_layer.affine)

## Save the corrected transform

In [ ]:
saved = save_manual_transform(exp, well, round_index, napari_affine)
print(f"Saved the corrected transform for {round_name} to {saved}")

## Optional - check the saved alignment

Shows every round of the well aligned with the saved transforms, including the one just fixed. Move the slider under the image to step through the rounds.

* Magenta = anchor round
* Green = current round
* White = overlap in alignment

In [ ]:
anchor, before, after = alignment_preview(exp, well, downscale_factor=4)

viewer = napari.Viewer()
viewer.add_image(anchor, blending="additive", colormap="magenta", name="Anchor Round")
viewer.add_image(after, blending="additive", colormap="green", name="After Alignment")
viewer.add_image(before, blending="additive", colormap="green", name="Before Alignment", visible=False)